In [25]:
import pandas as pd
import numpy as np
import yfinance as yf
import openpyxl
from scipy.interpolate import CubicSpline
from scipy.stats import zscore

In [26]:
df_clean = pd.DataFrame({'Date': pd.date_range(start='2016-01-01', end='2025-12-31', freq='W-WED')})
df_clean.head()

,Date
0,2016-01-06
1,2016-01-13
2,2016-01-20
3,2016-01-27
4,2016-02-03


 Baseline (weekly): GCPU

In [27]:
gcpu = pd.read_excel('GCPU.xlsx', sheet_name='GCPU_daily')
gcpu['GCPU_baseline'] = pd.to_numeric(gcpu['GCPU(PPP-Adjusted GDP)'], errors='coerce')
gcpu_wed = gcpu[gcpu['date'].dt.weekday == 2][['date', 'GCPU_baseline']]
df_clean = pd.merge(df_clean, gcpu_wed, left_on='Date', right_on='date', how='left').drop(columns=['date'])

df_clean.head()

,Date,GCPU_baseline
0,2016-01-06,93.447391
1,2016-01-13,43.149496
2,2016-01-20,56.229655
3,2016-01-27,83.497718
4,2016-02-03,153.977565


Clean energy equities (Log returns outcome):

    Primary: A European clean energy equity index or ETF(IQQH)

    Robustness: ICLN (global); TAN (solar); S&P Global Clean Energy Transition Index
    
    Placebo: STOXX 600 Industrials()

In [28]:
def get_return(file, col, name, skip=0):
    data = pd.read_csv(file, skiprows=skip)
    data.rename(columns={data.columns[0]: 'Date'}, inplace=True)
    data['Date'] = pd.to_datetime(data['Date'])
    data[col] = pd.to_numeric(data[col], errors='coerce')
    data[col] = data[col].ffill()
    data = data[data['Date'].dt.weekday == 2].sort_values('Date').dropna(subset=[col])
    data[name] = 100 * np.log(data[col]).diff()
    return data[['Date', name]]

df_clean = pd.merge(df_clean, get_return('clean_energy_etf_IQQH.csv', 'Close', 'y_IQQH_EUR', skip=[1,2]), on='Date', how='left') # Primary Euro
df_clean = pd.merge(df_clean, get_return('clean_energy_etf_ICLN.csv', 'Close', 'y_ICLN', skip=[1,2]), on='Date', how='left')    # Global
df_clean = pd.merge(df_clean, get_return('clean_energy_etf_TAN.csv', 'Close', 'y_TAN', skip=[1,2]), on='Date', how='left')      # Solar
stoxx = pd.read_excel('Libro1.xlsx', sheet_name='stoxx600 industrial') # Stoxx 600 Industrial
stoxx['y_stoxx'] = pd.to_numeric(stoxx['TRDPRC_1'], errors='coerce')
stoxx['y_stoxx'] = stoxx['y_stoxx'].ffill()
stoxx.rename(columns={'Timestamp': 'Date'}, inplace=True)
stoxx_wed = stoxx[stoxx['Date'].dt.weekday == 2][['Date', 'y_stoxx']]
stoxx_wed['y_stoxx'] = 100 * np.log(stoxx_wed['y_stoxx']).diff()
df_clean = pd.merge(df_clean, stoxx_wed, on='Date', how='left') 
idx_data = pd.read_excel(                                              # Global Clean Energy Transition Index
    'Global Clean Energy Transition Index.xls',
    skiprows=6
)
idx_data.columns = idx_data.columns.str.strip()
idx_data['Date'] = pd.to_datetime(
        idx_data['Effective date'],
        errors='coerce'
)
price_col = 'S&P Global Clean Energy Transition Index (USD)'
idx_data[price_col] = idx_data[price_col].ffill()
idx_wed = idx_data[idx_data['Date'].dt.weekday == 2].sort_values('Date').dropna(subset=[price_col]).copy()
idx_wed['y_Global_Clean_Index'] = 100 * np.log(idx_wed[price_col]).diff()
df_clean = pd.merge(df_clean, idx_wed[['Date', 'y_Global_Clean_Index']], on='Date', how='left')
df_clean.head()

,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index
0,2016-01-06,93.447391,-1.058602,-3.191225,-2.951626,NaN,NaN
1,2016-01-13,43.149496,-8.750428,-10.350249,-17.862594,-2.479420,NaN
2,2016-01-20,56.229655,-11.656931,-6.469321,-6.956787,-6.547708,NaN
3,2016-01-27,83.497718,7.120504,3.645616,6.026936,6.113354,NaN
4,2016-02-03,153.977565,-2.189532,2.008326,-1.308287,-3.123639,NaN


Climate policy uncertainty:

    Baseline (weekly): GCPU

    Robustness (monthly): CPU and decomposed CPU+/CPU−

        (i) step function (baseline)

        (ii) cubic spline interpolation

        (iii) MIDAS-style weighting (robustness)
    
    Falsification: General EPU


In [29]:
cpu_monthly = pd.read_csv('cpu_all_countries_monthly.csv')
cpu_monthly.drop(columns='cit', inplace=True)
cpu_monthly['year'] = cpu_monthly['year'].astype(int)
cpu_monthly['month'] = cpu_monthly['month'].astype(int)
cpu_monthly['ym'] = pd.to_datetime(
    cpu_monthly['year'].astype(str) + '-' 
    + cpu_monthly['month'].astype(str) + '-01'
).dt.to_period('M')

countries = ['CPU_DEU','CPU_FRA','CPU_ITA','CPU_ESP','CPU_IRL']

for c in countries:
    cpu_monthly[c] = zscore(
        cpu_monthly[c],
        nan_policy='omit'
    )
cpu_monthly['CPU_EU'] = cpu_monthly[['CPU_DEU', 'CPU_FRA', 'CPU_ITA', 'CPU_ESP', 'CPU_IRL']].mean(axis=1)

# step function
df_clean['ym'] = df_clean['Date'].dt.to_period('M')
df_clean = pd.merge(df_clean, cpu_monthly[['ym', 'CPU_EU']], on='ym', how='left')
df_clean = df_clean.rename(columns={'CPU_EU': 'CPU_EU_step'})

df_clean.head()


,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index,ym,CPU_EU_step
0,2016-01-06,93.447391,-1.058602,-3.191225,-2.951626,NaN,NaN,2016-01,0.252594
1,2016-01-13,43.149496,-8.750428,-10.350249,-17.862594,-2.479420,NaN,2016-01,0.252594
2,2016-01-20,56.229655,-11.656931,-6.469321,-6.956787,-6.547708,NaN,2016-01,0.252594
3,2016-01-27,83.497718,7.120504,3.645616,6.026936,6.113354,NaN,2016-01,0.252594
4,2016-02-03,153.977565,-2.189532,2.008326,-1.308287,-3.123639,NaN,2016-02,0.067218


In [30]:
#  cubic spline interpolation
cpu_monthly['Interpolation_Date'] = cpu_monthly['ym'].dt.to_timestamp()
full_daily = pd.DataFrame({'Date': pd.date_range(start=df_clean['Date'].min(), end=df_clean['Date'].max(), freq='D')})
full_daily = pd.merge(full_daily, cpu_monthly[['Interpolation_Date', 'CPU_EU']], left_on='Date', right_on='Interpolation_Date', how='left').drop(columns=['Interpolation_Date'])
full_daily = full_daily.set_index('Date')
full_daily['CPU_EU'] = full_daily['CPU_EU'].interpolate(method='cubicspline', limit_direction='both')
full_daily = full_daily.reset_index()
full_wed = full_daily[['Date', 'CPU_EU']].rename(columns={'CPU_EU': 'CPU_EU_spline'})
df_clean = pd.merge(df_clean, full_wed, on='Date', how='left')
# GEPU
gepu = pd.read_excel('Global_Economic_Policy_Uncertainty_EPU.xlsx')
gepu['ym'] = pd.to_datetime(gepu['Year'].astype(str) + '-' + gepu['Month'].astype(str) + '-01').dt.to_period('M')
gepu['GEPU_current'] = pd.to_numeric(gepu['GEPU_current'], errors='coerce')
gepu['GEPU_ppp'] = pd.to_numeric(gepu['GEPU_ppp'], errors='coerce')
gepu.head()
df_clean = pd.merge(df_clean, gepu[['ym', 'GEPU_current', 'GEPU_ppp']], on='ym', how='left')

EU ETS carbon prices: `FEUAc1`

In [31]:
carbon = pd.read_excel('Libro1.xlsx', sheet_name='FEUAc1')
carbon['EUA_Carbon'] = pd.to_numeric(carbon['SETTLE'], errors='coerce')
carbon.rename(columns={'Timestamp': 'Date'}, inplace=True)
carbon['l_t'] = 100*np.log(carbon['EUA_Carbon'])
carbon_wed = carbon[carbon['Date'].dt.weekday == 2].copy()
carbon_wed['c_t'] = carbon_wed['l_t'].diff()
df_clean = pd.merge(df_clean, carbon_wed[['Date', 'l_t', 'c_t']], on='Date', how='left')
df_clean.head()

,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index,ym,CPU_EU_step,CPU_EU_spline,GEPU_current,GEPU_ppp,l_t,c_t
0,2016-01-06,93.447391,-1.058602,-3.191225,-2.951626,NaN,NaN,2016-01,0.252594,-0.080481,144.526658,139.964254,204.769284,NaN
1,2016-01-13,43.149496,-8.750428,-10.350249,-17.862594,-2.479420,NaN,2016-01,0.252594,-0.011618,144.526658,139.964254,197.962121,-6.807164
2,2016-01-20,56.229655,-11.656931,-6.469321,-6.956787,-6.547708,NaN,2016-01,0.252594,0.034152,144.526658,139.964254,184.054963,-13.907157
3,2016-01-27,83.497718,7.120504,3.645616,6.026936,6.113354,NaN,2016-01,0.252594,0.059720,144.526658,139.964254,177.495235,-6.559728
4,2016-02-03,153.977565,-2.189532,2.008326,-1.308287,-3.123639,NaN,2016-02,0.067218,0.067980,147.483002,144.826617,172.455072,-5.040163


Financial conditions and energy price controls:

    Credit spreads: ICE BofA US HY OAS.

    European iTraxx Crossover: UBS long ITRAXX main 5Y index excess return EUR.

    Rates: Euro area 2Y/10Y yields and the term spread.

    Equity volatility: VSTOXX to separate CPU-specific effects from broad market stress.

    Energy prices: TTF natural gas returns and Brent crude oil returns, given their role in EUA dynamics via fuel switching.

In [32]:
itraxx = pd.read_excel('Libro2.xlsx')                                   # Itraxx
itraxx['itraxx'] = pd.to_numeric(itraxx['TRDPRC_1'], errors='coerce')
itraxx.rename(columns={'Timestamp': 'Date'}, inplace=True)
itraxx_wed = itraxx[itraxx['Date'].dt.weekday == 2].copy()
itraxx_wed['itraxx_robustness'] = itraxx_wed['itraxx'].diff()
df_clean = pd.merge(df_clean, itraxx_wed[['Date', 'itraxx_robustness']], on='Date', how='left')

r2 = pd.read_csv('rate_2yield.csv', parse_dates=['observation_date'])    # 2-year yield
r10 = pd.read_csv('rate_10yield.csv', parse_dates=['observation_date'])  # 10-year yield
rates = pd.merge(r2, r10, on='observation_date')
rates['rate_2y'] = pd.to_numeric(rates['IR3TIB01EZM156N'], errors='coerce')
rates['rate_10y'] = pd.to_numeric(rates['IRLTLT01EZM156N'], errors='coerce')
rates['Term_Spread'] = rates['rate_10y'] - pd.to_numeric(rates['rate_2y'], errors='coerce')
rates['ym'] = rates['observation_date'].dt.to_period('M')
df_clean = pd.merge(df_clean, rates[['ym', 'rate_10y', 'Term_Spread']], on='ym', how='left')  # Rates and term spread
v2tx = pd.read_csv('v2tx.txt', sep=';', parse_dates=['Date'], dayfirst=True)    # VSTOXX
v2tx.head()
v2tx['log_VSTOXX'] = 100*np.log(pd.to_numeric(v2tx['Indexvalue'], errors='coerce'))
df_clean = pd.merge(df_clean, v2tx[v2tx['Date'].dt.weekday == 2][['Date', 'log_VSTOXX']], on='Date', how='left')
df_clean = pd.merge(df_clean, get_return('ICE Dutch TTF Natural Gas Futures Historical Data.csv', 'Price', 'TTF_return'), # Natural Gas
                     on='Date', how='left')
df_clean = pd.merge(df_clean, get_return('DCOILBRENTEU.csv', 'DCOILBRENTEU', 'Brent_return'), on='Date', how='left')  # Brent Oil
df_clean.drop(columns=['ym'], inplace=True)
df_clean.head()

,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index,CPU_EU_step,CPU_EU_spline,GEPU_current,GEPU_ppp,l_t,c_t,itraxx_robustness,rate_10y,Term_Spread,log_VSTOXX,TTF_return,Brent_return
0,2016-01-06,93.447391,-1.058602,-3.191225,-2.951626,NaN,NaN,0.252594,-0.080481,144.526658,139.964254,204.769284,NaN,NaN,1.114501,1.260651,326.991109,1.431592,-5.062916
1,2016-01-13,43.149496,-8.750428,-10.350249,-17.862594,-2.479420,NaN,0.252594,-0.011618,144.526658,139.964254,197.962121,-6.807164,-0.235972,1.114501,1.260651,333.700724,-6.514693,-17.041281
2,2016-01-20,56.229655,-11.656931,-6.469321,-6.956787,-6.547708,NaN,0.252594,0.034152,144.526658,139.964254,184.054963,-13.907157,-0.611980,1.114501,1.260651,355.857998,-10.205742,-9.422609
3,2016-01-27,83.497718,7.120504,3.645616,6.026936,6.113354,NaN,0.252594,0.059720,144.526658,139.964254,177.495235,-6.559728,0.365467,1.114501,1.260651,332.443152,3.734954,20.192816
4,2016-02-03,153.977565,-2.189532,2.008326,-1.308287,-3.123639,NaN,0.067218,0.067980,147.483002,144.826617,172.455072,-5.040163,-0.615157,1.041831,1.225402,342.546389,-2.747426,1.713171


Dummy variables

In [33]:
df_clean['COVID_dummy'] = (
    (df_clean['Date'] >= '2020-02-01') &
    (df_clean['Date'] <= '2023-06-30')
).astype(int)

df_clean['Energy_crisis_dummy'] = (
    (df_clean['Date'] >= '2021-01-01') &
    (df_clean['Date'] <= '2022-12-31')
).astype(int)

df_clean.head()

,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index,CPU_EU_step,CPU_EU_spline,GEPU_current,...,l_t,c_t,itraxx_robustness,rate_10y,Term_Spread,log_VSTOXX,TTF_return,Brent_return,COVID_dummy,Energy_crisis_dummy
0,2016-01-06,93.447391,-1.058602,-3.191225,-2.951626,NaN,NaN,0.252594,-0.080481,144.526658,...,204.769284,NaN,NaN,1.114501,1.260651,326.991109,1.431592,-5.062916,0,0
1,2016-01-13,43.149496,-8.750428,-10.350249,-17.862594,-2.479420,NaN,0.252594,-0.011618,144.526658,...,197.962121,-6.807164,-0.235972,1.114501,1.260651,333.700724,-6.514693,-17.041281,0,0
2,2016-01-20,56.229655,-11.656931,-6.469321,-6.956787,-6.547708,NaN,0.252594,0.034152,144.526658,...,184.054963,-13.907157,-0.611980,1.114501,1.260651,355.857998,-10.205742,-9.422609,0,0
3,2016-01-27,83.497718,7.120504,3.645616,6.026936,6.113354,NaN,0.252594,0.059720,144.526658,...,177.495235,-6.559728,0.365467,1.114501,1.260651,332.443152,3.734954,20.192816,0,0
4,2016-02-03,153.977565,-2.189532,2.008326,-1.308287,-3.123639,NaN,0.067218,0.067980,147.483002,...,172.455072,-5.040163,-0.615157,1.041831,1.225402,342.546389,-2.747426,1.713171,0,0


In [34]:
covariate_cols = [
    'GCPU_baseline', 'l_t', 'c_t', 
    'itraxx_robustness', 
    'CPU_EU_step','CPU_EU_spline','GEPU_current','GEPU_ppp',
    'rate_10y', 'Term_Spread', 'log_VSTOXX', 'TTF_return', 'Brent_return'
]
for col in covariate_cols:
    df_clean[col] = df_clean[col].shift(1)

for col in covariate_cols:
    mean = df_clean[col].mean()
    std = df_clean[col].std()
    df_clean[col] = (df_clean[col] - mean) / std
df_clean.head()

,Date,GCPU_baseline,y_IQQH_EUR,y_ICLN,y_TAN,y_stoxx,y_Global_Clean_Index,CPU_EU_step,CPU_EU_spline,GEPU_current,...,l_t,c_t,itraxx_robustness,rate_10y,Term_Spread,log_VSTOXX,TTF_return,Brent_return,COVID_dummy,Energy_crisis_dummy
0,2016-01-06,NaN,-1.058602,-3.191225,-2.951626,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,2016-01-13,-0.635013,-8.750428,-10.350249,-17.862594,-2.479420,NaN,-0.284724,-0.609910,-1.112275,...,-1.318898,NaN,NaN,-0.391668,0.532892,1.124088,0.131930,-0.738335,0,0
2,2016-01-20,-0.994222,-11.656931,-6.469321,-6.956787,-6.547708,NaN,-0.284724,-0.609909,-1.112275,...,-1.386152,-1.141352,-0.921037,-0.391668,0.532892,1.351866,-0.669060,-2.447551,0,0
3,2016-01-27,-0.900808,7.120504,3.645616,6.026936,6.113354,NaN,-0.284724,-0.609909,-1.112275,...,-1.523553,-2.254918,-2.231922,-0.391668,0.532892,2.104062,-1.041121,-1.360428,0,0
4,2016-02-03,-0.706070,-2.189532,2.008326,-1.308287,-3.123639,NaN,-0.284724,-0.609908,-1.112275,...,-1.588362,-1.102544,1.175768,-0.391668,0.532892,1.309174,0.364110,2.865456,0,0


In [35]:
df_clean = df_clean.drop(df_clean.index[0])
df_clean.to_csv(
    'final_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)


In [36]:
#Please make a small table showing, for each variable: start date, end date, and % missing.
vars_list = [
    'GCPU_baseline',
    'y_IQQH_EUR',
    'y_ICLN',
    'y_TAN',
    'y_stoxx',
    'y_Global_Clean_Index',
    'CPU_EU_step',
    'CPU_EU_spline',
    'GEPU_current',
    'GEPU_ppp',
    'l_t',
    'c_t',
    'itraxx_robustness',
    'rate_10y',
    'Term_Spread',
    'log_VSTOXX',
    'TTF_return',
    'Brent_return',
    'COVID_dummy',
    'Energy_crisis_dummy'
]

rows = []
for col in vars_list:
    s = df_clean[['Date', col]].dropna(subset=[col])
    if len(s) == 0:
        start_date = None
        end_date = None
    else:
        start_date = s['Date'].min().date()
        end_date = s['Date'].max().date()

    rows.append({
        'Variable': col,
        'Start date': start_date,
        'End date': end_date,
        'Missing %': round(df_clean[col].isna().mean() * 100, 2)
    })

df_table = pd.DataFrame(rows)
df_table.head()
print(df_table.to_latex(index=False))

\begin{tabular}{lllr}
\toprule
Variable & Start date & End date & Missing % \\
\midrule
GCPU_baseline & 2016-01-13 & 2025-12-31 & 0.000000 \\
y_IQQH_EUR & 2016-01-13 & 2025-12-17 & 1.920000 \\
y_ICLN & 2016-01-13 & 2025-12-31 & 1.340000 \\
y_TAN & 2016-01-13 & 2025-12-31 & 1.340000 \\
y_stoxx & 2016-01-13 & 2025-12-31 & 0.960000 \\
y_Global_Clean_Index & 2016-05-11 & 2025-12-31 & 3.650000 \\
CPU_EU_step & 2016-01-13 & 2020-01-01 & 60.080000 \\
CPU_EU_spline & 2016-01-13 & 2025-12-31 & 0.000000 \\
GEPU_current & 2016-01-13 & 2025-12-03 & 0.770000 \\
GEPU_ppp & 2016-01-13 & 2025-12-03 & 0.770000 \\
l_t & 2016-01-13 & 2025-12-24 & 2.110000 \\
c_t & 2016-01-20 & 2025-12-24 & 2.300000 \\
itraxx_robustness & 2016-01-20 & 2025-12-31 & 1.540000 \\
rate_10y & 2016-01-13 & 2025-12-31 & 0.000000 \\
Term_Spread & 2016-01-13 & 2025-12-31 & 0.000000 \\
log_VSTOXX & 2016-01-13 & 2025-12-24 & 1.540000 \\
TTF_return & 2016-01-13 & 2025-12-31 & 0.960000 \\
Brent_return & 2016-01-13 & 2025-12-31 & 0.0000